In [94]:
# Reset memoire
%reset -f 
# Affiche fenêtre separée avec interactivité
%matplotlib qt5
import numpy as np
from scipy.io import wavfile
import matplotlib.pyplot as plt
# Fermer toutes les fenetres graphiques
plt.close('all')
import sounddevice as sd

In [95]:
def computeFFT(signal, shift):
    # Calcul de la transformée de Fourier
    nbSamples = len(signal)
    fftSignal = np.fft.fft(signal)/nbSamples
    # On shifte pour le bilatéral
    if shift : fftSignal = np.fft.fftshift(fftSignal)
    # Calcul du module
    magfftSignal = np.abs(fftSignal);
    # Calcul de la phase
    phasefftSignal = np.angle(fftSignal);
    return magfftSignal, phasefftSignal;

def plotFFT(f, magfftSignal, phasefftSignal, label):
    # Affichage des spectres de module et de phase
    plt.figure()
    # Affichage du module
    plt.subplot(2,1,1);
    plt.plot(f, magfftSignal);
    plt.title('Spectre de module de ' + label);
    plt.xlabel('fréquence (Hz)');
    plt.ylabel('Amplitude');
    # Affichage de la phase
    plt.subplot(2,1,2);
    plt.plot(f, phasefftSignal);
    plt.title('Spectre de phase de ' + label);
    plt.xlabel('fréquence (Hz)');
    plt.ylabel('Phase');

# Calcul du spectre monolatéral (freq > 0)
def computeSingleSidedFFT(Fe, signal) :
    # Nombre d'échantillons et résolution fréquentielle
    nbSamples = len(signal);
    df = Fe/nbSamples;
    # Construction du vecteur fréquentiel
    if nbSamples % 2 == 1: # regarde si echantillon est impaire ou pas reste division entiere
        # nbSamples impair
        N = (nbSamples + 1)//2 # pour rester ccentrer sur 0
        f = np.arange(0, Fe/2 + df/2, df ) # df/2 decalage mon signal d1/2 resolution freq
    else:
        # nbSamples pair
        N = nbSamples//2
        f = np.arange(0, Fe/2, df )
    magfftSignal, phasefftSignal = computeFFT(signal, False)
    # Arrangement pour obtenir les bonnes amplitudes
    magfftSignal = np.concatenate((magfftSignal[0], 2*magfftSignal[1:N]), axis = None)
    phasefftSignal = phasefftSignal[0:N]
    return f, magfftSignal, phasefftSignal

# Calcul du spectre bilatéral (freq > 0 et < 0)
def computeDoubleSidedFFT(Fe, signal) :
    # Nombre d'échantillons et résolution fréquentielle
    nbSamples = len(signal);
    df = Fe/nbSamples;
    # Construction du vecteur fréquentiel
    if nbSamples % 2 == 1: # regarde si echantillon est impaire ou pas reste division entiere
        # nbSamples impair
        f = np.arange(-Fe/2 + df/2, Fe/2 + df/2, df ) # df/2 decalage mon signal d1/2 resolution freq
    else:
        # nbSamples pair
        f = np.arange(-Fe/2, Fe/2, df )
    # Calcul de la transformée de Fourier
    magfftSignal, phasefftSignal = computeFFT(signal, True)
    return f, magfftSignal, phasefftSignal

In [96]:
# Q.1 Lecture du signal
Fe, note = wavfile.read('note.wav');
note = note/np.max(np.abs(note))
#sd.play(note, Fe)
#sd.wait()
# Q.2 Affichage spectre monolatéral note brute
f, magfftSignal, phasefftSignal = computeSingleSidedFFT(Fe, note)
plotFFT(f, magfftSignal, phasefftSignal, 'la note (monolatéral)')
# On trouve et on entend des parasites du secteur (50Hz) et à 24.2Hz

/var/folders/bq/rp1pp2j539s265s4psl_kbtw0000gn/T/ipykernel_2173/569790985.py:2: WavFileWarning: Chunk (non-data) not understood, skipping it.
  Fe, note = wavfile.read('note.wav');


In [121]:
# Q.3
noiseLevel = 0.05
noisyNote = note + noiseLevel*np.random.normal(size=len(note))
sd.play(noisyNote, Fe)
sd.wait()
# Affichage spectre monolatéral note bruitée
f, magfftSignal, phasefftSignal = computeSingleSidedFFT(Fe, noisyNote)
plotFFT(f, magfftSignal, phasefftSignal, 'la note bruitée (monolatéral)')
# On trouve et on entend des parasites du secteur (50Hz) et à 24.2Hz

In [119]:
# Q.4 : filtrage avec un RIF (IIR) type Butterworth
# Il n'existe pas de "formule magique" permettant de définir la 
# fréquence de coupure (fc) et l'ordre (order) d'un filtre. 
# C'est en fait très souvent une affaire de bon sens, de compromis
# et peut dépendre de la situation ! 
# Nous allons ici utiliser un filtre passe bas pour 
# supprimer le bruit ajouté qui se situe aux hautes fréquences en 
# grande majorité. On sait que la fondamentale de la note est environ
# à 83Hz et que les autres harmoniques sont importants. 
# Il va falloir choisir une fc pas trop basse pour ne pas affaiblir 
# la richesse de la note mais pas trop haute pour ne pas affaiblir le
# filtrage. Pour l'ordre il ne sert à rien de l'augmenter indéfiniement
# dans la mesure ou la complexité des calculs augmente avec lui et que 
# les distorsions de phase deviennent trop importantes.
import scipy.signal as signal
order = 5
fc = 300
b, a = signal.butter(order, fc/(Fe/2), 'low')
iirFilteredNoisyNote = signal.lfilter(b, a, noisyNote)
sd.play(iirFilteredNoisyNote, Fe)
sd.wait()

In [120]:
# Q.5 : filtrage avec un RIF (FIR)
b = signal.remez(order, [0, fc, fc + 10, Fe/2], [1, 0], fs=Fe)
firFilteredNoisyNote =  signal.lfilter(b, 1, noisyNote)
sd.play(firFilteredNoisyNote, Fe)
sd.wait()

In [ ]:
# Q.6
# Pour un même ordre les filtres RII sont plus
# performants que les filtres RIF. L'avantage des
# filtres RIF vient du fait qu'ils sont à phase 
# linéaire : le retard de phase est constant
# (toutes les ondes sont retardées de la même
# quantité). C'est une caractéristique importante
# dans le domaine audio.
# Il est possible de supprimer une grande partie
# du bruit qui se situe en haute fréquence sans
# détruire la note. Mais il est impossible de le
# supprimer totalement parce qu'une partie de ce
# bruit est situé aux mêmes fréquences que celles qui
# composent la note. Donc supprimer ces composantes
# fréquentielles a pour conséquence la suppression
# d'une partie des fréquences de la note. C'est une
# des limites du filtrage fréquentiel : si le bruit
# et l'information sont aux mêmes fréquences il sera
# impossible de les séparer par filtrage fréquentiel.